# 07 Features carrying signal & model failures
By Tahreem Karim

Uses the folds and the modal hyperparameters from notebook 05, so every number here is paired with
Table A2. Hyperparameters are held fixed (no re-tuning) so differences come from the features alone. The notebook is organized by questions interrogating the feature groups. 

#### Table of contents:
1. Which features is the model paying attention to? 
- **SHAP** for XGBoost fit on all counties (Figure A1), and each feature group's share of the model's attention.
2. Does performance drop if a feature group is removed & how well does that group do on its own? 
- **Group ablation** (Table A3): drop each group, or keep only that group.
3. Which specific policy feature is doing work?
- **Which part of policy?** Each policy dimension added alone to infrastructure + geography.
4. Could the policy signal just be random luck (not real)?
- **State level permutation test**: shuffle whole policy profiles btw. states. Any state level
   variable can help a little by chance, but real policy features have to beat shuffled ones.
5. Does result hold up if small choices change (outliers, which features are included, how counties are split, a demand stand-in)?
- **Sensitivity** (Table A4): electricity outliers, a dropped feature, random vs state grouped Cross Val, & a
   demand proxy (population) that none of our sources carries.
6. Would different model settings have a different story?
- **Hyperparameter sensitivity** (Figure A2).
7. Where does the model get it wrong, why?
- **Failure analysis**: the six hypotheses in the report, tested on 'out of fold predictions', and three
   specific counties explained w/ fold models that scored them (Figure A3).

Figures from the CSVs referenced in this ntbk are drawn in notebook 10. 

In [ ]:
import json
import pickle
import warnings
import geopandas as gpd
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", message="IProgress not found")   # tqdm notice printed when shap is imported
import shap
from scipy.stats import spearmanr
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold

from pipeline_config import (FEATURE_GROUP, FEATURE_SETS, GEO, GROUP, INFRA_GROUPS, POLICY, POLICY_GROUPS, ROOT,
                             STAGE4, TARGET, fixed_model, load_model_table, outer_splits)

warnings.filterwarnings("ignore", category=UserWarning)
RERUN = False            # `True` recomputes slow parts: permutation test, hyperparameter grid

d = load_model_table()
y, groups = d[TARGET].values, d[GROUP].values
SPW = (y == 0).sum() / y.sum()
SPLITS = list(outer_splits(y, groups))            # the same 15 folds as notebook 05
FULL = FEATURE_SETS["+ AI policy"]
GEO_SET = FEATURE_SETS["+ Geography"]
modal = json.loads((STAGE4 / "modal_params.json").read_text())
INT = ("max_depth", "n_estimators", "n_neighbors", "min_samples_leaf")
cast = lambda p: {k: (None if v in ("None", "nan") else int(float(v)) if k in INT else float(v) if k in ("C", "learning_rate")
                      else v) for k, v in p.items()}

PARAMS = {fam: cast(p) for fam, p in modal.items()}
print("modal hyperparameters:", PARAMS)


def cv_scores(cols, family="XGBoost", data=None, splits=None, cap=0.99, params=None):
    """PR-AUC on each of the 15 state-grouped test folds, hyperparameters fixed."""
    data = d if data is None else data
    out = []
    for seed, k, tr, te in (splits or SPLITS):
        m = fixed_model(family, cols, SPW, params or PARAMS[family], cap=cap).fit(data[cols].iloc[tr], y[tr])
        out.append(average_precision_score(y[te], m.predict_proba(data[cols].iloc[te])[:, 1]))
    return np.array(out)


ms = lambda s: f"{s.mean():.3f} ± {s.std(ddof=1):.3f}"          # sample SD, notebook 05
FAMILIES = ["XGBoost", "Logistic regression"]
base = {fam: {name: cv_scores(cols, fam) for name, cols in FEATURE_SETS.items()} for fam in FAMILIES}
pd.DataFrame([{"family": f, "set": s, "pr_auc_mean": v.mean(), "pr_auc_sd": v.std(ddof=1)} for f, r in base.items()
              for s, v in r.items()]).to_csv(STAGE4 / "fixed_param_scores.csv", index=False)

for fam, r in base.items():
    print(fam, {k: ms(v) for k, v in r.items()})

modal hyperparameters: {'Logistic regression': {'C': 0.01}, 'k-nearest neighbors': {'n_neighbors': 25, 'weights': 'distance'}, 'Balanced random forest': {'max_depth': 6, 'min_samples_leaf': 5}, 'XGBoost': {'max_depth': 2, 'learning_rate': 0.03, 'n_estimators': 200}}


XGBoost {'Infrastructure': '0.372 ± 0.060', '+ Geography': '0.376 ± 0.054', '+ AI policy': '0.378 ± 0.065'}
Logistic regression {'Infrastructure': '0.343 ± 0.068', '+ Geography': '0.358 ± 0.065', '+ AI policy': '0.342 ± 0.056'}


## 1. SHAP: model's attention shares (Figure A1)

XGBoost with the modal hyperparameters, fit on all 3,099 counties with the full feature set. SHAP values
are in log odds units of the raw (uncalibrated) model.

In [ ]:
full_model = fixed_model("XGBoost", FULL, SPW, PARAMS["XGBoost"]).fit(d[FULL], y)
Xp = pd.DataFrame(full_model.named_steps["prep"].transform(d[FULL]), columns=FULL)
sv = shap.TreeExplainer(full_model.named_steps["m"]).shap_values(Xp)
attention = (pd.DataFrame(np.abs(sv), columns=FULL).mean().rename("mean_abs_shap").to_frame()
             .assign(group=lambda t: t.index.map(FEATURE_GROUP)))

# direction: does a higher feature value push the prediction up (+) or down (-)? Spearman values vs SHAP
# features the model never splits on have all zero SHAP, so they get no direction (NaN)
attention["direction"] = [spearmanr(Xp[c], sv[:, i]).statistic if Xp[c].nunique() > 1 and np.ptp(sv[:, i]) > 0 else np.nan
                          for i, c in enumerate(FULL)]
group_share = (attention.groupby("group").mean_abs_shap.sum() / attention.mean_abs_shap.sum()).sort_values()
attention.sort_values("mean_abs_shap", ascending=False).to_csv(STAGE4 / "shap_importance.csv")
group_share.rename("share").to_csv(STAGE4 / "shap_group_share.csv")
pd.concat([d[["county_fips"]], pd.DataFrame(sv, columns=[f"shap__{c}" for c in FULL]),
           Xp.add_prefix("value__")], axis=1).to_csv(STAGE4 / "shap_full.csv", index=False)

print(group_share.round(3))
attention.sort_values("mean_abs_shap", ascending=False).head(10).round(3)

group
Cooling                        0.016
Electricity cost               0.061
Geography                      0.103
AI policy                      0.121
Grid access and suitability    0.304
Land and taxes                 0.395
Name: mean_abs_shap, dtype: float32


,mean_abs_shap,group,direction
land_value_dollar_per_sqft_mean,0.706,Land and taxes,0.909
composite_siting_suitability_mean,0.350,Grid access and suitability,0.953
distance_to_substation_mean,0.244,Grid access and suitability,-0.573
pol_share_rights,0.156,AI policy,0.212
div_mountain,0.129,Geography,0.497
commercial_electricity_rates_dollar_per_kwh_mean,0.089,Electricity cost,0.692
real_property_tax_rate_mean,0.074,Land and taxes,0.898
div_pacific,0.063,Geography,0.351
pol_share_public,0.054,AI policy,0.133
sales_tax_rate_mean,0.049,Land and taxes,0.074


>Land value/taxes (40%) and grid access/suitability (30%) dominate model’s attention, together accounting for about 70% of it. AI policy sits in the middle at 12%, just ahead of geography (10%) & electricity cost (6%), cooling (2%)

## 2 Group ablation (Table A3)

In [ ]:
GROUPS = {**INFRA_GROUPS, "Geography": GEO, "AI policy": POLICY}
abl = [{"group": "All features", "features": len(FULL), "removed": ms(base["XGBoost"]["+ AI policy"]),
        "removed_mean": base["XGBoost"]["+ AI policy"].mean(), "alone": "", "alone_mean": np.nan}]

for name, cols in GROUPS.items():
    rem, alone = cv_scores([c for c in FULL if c not in cols]), cv_scores(cols)
    abl.append({"group": name, "features": len(cols), "removed": ms(rem), "removed_mean": rem.mean(),
                "alone": ms(alone), "alone_mean": alone.mean()})

ablation = pd.DataFrame(abl)
ablation.to_csv(STAGE4 / "group_ablation.csv", index=False)
ablation

,group,features,removed,removed_mean,alone,alone_mean
0,All features,29,0.378 ± 0.065,0.378226,,NaN
1,Electricity cost,2,0.390 ± 0.065,0.389879,0.081 ± 0.017,0.081122
2,Grid access and suitability,4,0.314 ± 0.061,0.313921,0.258 ± 0.059,0.257526
3,Land and taxes,4,0.251 ± 0.057,0.251092,0.325 ± 0.058,0.324565
4,Cooling,1,0.383 ± 0.063,0.382967,0.090 ± 0.011,0.090118
5,Geography,9,0.368 ± 0.068,0.368246,0.122 ± 0.025,0.121510
6,AI policy,9,0.376 ± 0.054,0.375633,0.094 ± 0.028,0.093877


>Policy adds real but small value. Land and taxes is most important group: removing it drops PR-AUC the most (0.378 → 0.251), and does nearly as well as full model on its own (0.325). Grid access and suitability is close second. AI policy and geography barely move the needle when removed (0.378 → 0.376 and 0.368) and are weak on their own (0.094 & 0.122)

## 3 Which part of policy? Each dimension added alone to infrastructure + geography

In [4]:
dims = []
for fam in FAMILIES:
    ref = base[fam]["+ Geography"]
    for name, cols in {**POLICY_GROUPS, "All policy features": POLICY}.items():
        s = cv_scores(GEO_SET + cols, fam)
        dims.append({"family": fam, "dimension": name, "pr_auc": ms(s), "gain_vs_geography": (s - ref).mean(),
                     "gain_sd": (s - ref).std(ddof=1), "folds_improved": int(((s - ref) > 0).sum())})
dimensions = pd.DataFrame(dims)
dimensions.to_csv(STAGE4 / "policy_dimensions.csv", index=False)
dimensions.round(3)

,family,dimension,pr_auc,gain_vs_geography,gain_sd,folds_improved
0,XGBoost,Attention and follow-through,0.375 ± 0.061,-0.000,0.017,6
1,XGBoost,Timing,0.378 ± 0.058,0.003,0.013,10
2,XGBoost,Content,0.378 ± 0.061,0.002,0.027,7
3,XGBoost,Infrastructure bills,0.377 ± 0.055,0.001,0.009,7
4,XGBoost,All policy features,0.378 ± 0.065,0.003,0.032,8
5,Logistic regression,Attention and follow-through,0.351 ± 0.060,-0.007,0.016,5
6,Logistic regression,Timing,0.357 ± 0.065,-0.002,0.003,5
7,Logistic regression,Content,0.357 ± 0.063,-0.002,0.012,8
8,Logistic regression,Infrastructure bills,0.354 ± 0.060,-0.004,0.012,8
9,Logistic regression,All policy features,0.342 ± 0.056,-0.016,0.029,4


>For XGBoost none of individual policy dimensions (timing, content, infrastructure bills, attention/follow-through) meaningfully help, all gains are tiny (0.001–0.003). Even all of them combined barely move the needle. For logistic regression, adding policy features actually hurts slightly.

## 4 State-level permutation test

Shuffle which state gets which policy profile (the nine columns move together, and every county in a state
keeps one shared profile), refit, and record the gain over + Geography. Repeated 100 times per family.

In [5]:
perm_file = STAGE4 / "permutation_test.csv"
if RERUN or not perm_file.exists():
    from joblib import Parallel, delayed
    profile = d.groupby(GROUP)[POLICY].first()
    orders = [np.random.default_rng(696 + b).permutation(profile.index.values) for b in range(100)]

    def one_permutation(b):
        shuffled = profile.copy()
        shuffled.index = orders[b]
        dp = d.drop(columns=POLICY).join(shuffled, on=GROUP)
        return [{"perm": b, "family": fam, "gain": (cv_scores(FULL, fam, data=dp) - base[fam]["+ Geography"]).mean()}
                for fam in FAMILIES]

    null = Parallel(n_jobs=2)(delayed(one_permutation)(b) for b in range(100))
    pd.DataFrame([r for rows in null for r in rows]).to_csv(perm_file, index=False)
null = pd.read_csv(perm_file)
perm = []
for fam in FAMILIES:
    obs = (base[fam]["+ AI policy"] - base[fam]["+ Geography"]).mean()
    nd = null[null.family == fam].gain
    perm.append({"family": fam, "observed_gain": obs, "null_mean": nd.mean(), "null_95th": nd.quantile(0.95),
                 "p_value": (1 + (nd >= obs).sum()) / (len(nd) + 1), "permutations": len(nd)})
permutation = pd.DataFrame(perm)
permutation.to_csv(STAGE4 / "permutation_summary.csv", index=False)
permutation.round(4)

,family,observed_gain,null_mean,null_95th,p_value,permutations
0,XGBoost,0.0026,-0.0064,0.0059,0.1089,100
1,Logistic regression,-0.0164,-0.0118,0.0091,0.6337,100


>XGBoost: small positive effect (p=0.11) not statistically distinguishable from what shuffled, fake policy data produces by chance. For logistic regression, the effect is negative. Neither model can claim “this isn’t noise.”

## 5 Sensitivity (Table A4)

Does it hold up under small changes. Rows 2-3 change a solution element; row 4 changes the validation; row 5 adds a demand proxy (county
population and density, Census 2024) that is outside our design, to see whether policy is standing in for demand.

In [ ]:
def random_splits(seeds=(0, 1, 2)):
    for s in seeds:
        for k, (tr, te) in enumerate(StratifiedKFold(5, shuffle=True, random_state=s).split(np.zeros(len(y)), y)):
            yield s, k, tr, te


d["log_population"] = np.log10(d.population_2024)
d["log_density"] = np.log10(d.pop_density)
DEMAND = ["log_population", "log_density"]

sens = []
for fam in FAMILIES:
    rows = {
        "Electricity capped at the 99th percentile (default)": (base[fam]["+ Geography"], base[fam]["+ AI policy"]),
        "Electricity outliers left raw": (cv_scores(GEO_SET, fam, cap=None), cv_scores(FULL, fam, cap=None)),
        "Commercial electricity dropped": tuple(cv_scores([c for c in cols if not c.startswith("commercial")], fam)
                                                for cols in (GEO_SET, FULL)),
        "Random (not state-grouped) CV": tuple(cv_scores(cols, fam, splits=list(random_splits())) for cols in (GEO_SET, FULL)),
        "+ population and density (demand proxy)": tuple(cv_scores(cols + DEMAND, fam) for cols in (GEO_SET, FULL))}
    
    for name, (g_s, p_s) in rows.items():
        sens.append({"family": fam, "variation": name, "geography_set": ms(g_s), "policy_set": ms(p_s),
                     "geography_set_mean": g_s.mean(), "policy_set_mean": p_s.mean(),
                     "policy_gain": (p_s - g_s).mean(), "gain_sd": (p_s - g_s).std(ddof=1)})

sensitivity = pd.DataFrame(sens)
sensitivity.to_csv(STAGE4 / "sensitivity.csv", index=False)
sensitivity.round(3)

,family,variation,geography_set,policy_set,geography_set_mean,policy_set_mean,policy_gain,gain_sd
0,XGBoost,Electricity capped at the 99th percentile (def...,0.376 ± 0.054,0.378 ± 0.065,0.376,0.378,0.003,0.032
1,XGBoost,Electricity outliers left raw,0.375 ± 0.055,0.380 ± 0.065,0.375,0.380,0.005,0.032
2,XGBoost,Commercial electricity dropped,0.384 ± 0.055,0.382 ± 0.064,0.384,0.382,-0.002,0.021
3,XGBoost,Random (not state-grouped) CV,0.380 ± 0.087,0.376 ± 0.084,0.380,0.376,-0.004,0.021
4,XGBoost,+ population and density (demand proxy),0.533 ± 0.047,0.519 ± 0.044,0.533,0.519,-0.014,0.024
5,Logistic regression,Electricity capped at the 99th percentile (def...,0.358 ± 0.065,0.342 ± 0.056,0.358,0.342,-0.016,0.029
6,Logistic regression,Electricity outliers left raw,0.360 ± 0.072,0.342 ± 0.064,0.360,0.342,-0.019,0.031
7,Logistic regression,Commercial electricity dropped,0.361 ± 0.065,0.347 ± 0.059,0.361,0.347,-0.014,0.028
8,Logistic regression,Random (not state-grouped) CV,0.370 ± 0.057,0.373 ± 0.054,0.370,0.373,0.003,0.011
9,Logistic regression,+ population and density (demand proxy),0.553 ± 0.036,0.546 ± 0.038,0.553,0.546,-0.007,0.020


>Mostly yes. Electricity outliers handling and random vs. state-grouped splits barely move policy effect. One big exception: adding population/density as a demand stand-in massively boosts overall performance (0.38 → 0.53 PR-AUC for XGBoost). Bigger places are more likely to get sited, but even then, policy’s contribution is small

## 6 Hyperparameter sensitivity (Figure A2)

XGBoost, full feature set, 500 trees: PR-AUC over depth × learning rate on the same 15 folds.

In [ ]:
grid_file = STAGE4 / "hyperparameter_grid.csv"
if RERUN or not grid_file.exists():
    cells = []
    for depth in [2, 3, 4, 5, 6]:
        for lr in [0.01, 0.03, 0.1, 0.3]:
            s = cv_scores(FULL, params={"max_depth": depth, "learning_rate": lr, "n_estimators": 500})
            cells.append({"max_depth": depth, "learning_rate": lr, "mean": s.mean(), "sd": s.std()})
    pd.DataFrame(cells).to_csv(grid_file, index=False)

hgrid = pd.read_csv(grid_file)
hgrid.pivot(index="max_depth", columns="learning_rate", values="mean").round(3)

learning_rate,0.01,0.03,0.10,0.30
max_depth,,,,
2,0.386,0.366,0.315,0.283
3,0.384,0.350,0.303,0.278
4,0.376,0.345,0.307,0.287
5,0.365,0.339,0.310,0.303
6,0.363,0.337,0.306,0.275


>Slow learning with shallow trees is best (up to 0.386 PR-AUC at depth 2, learning rate 0.01). Faster learning hurts across every depth (down to ~0.28)

## 7 Failure analysis

"Out of fold" calibrated probabilities from notebook 05 (XGBoost), averaged over the three seeds. Each
county's SHAP values come from the three fold-models that held out its state, not from a model that saw it.

In [ ]:
oof = pd.read_csv(STAGE4 / "oof_predictions.csv", dtype={"county_fips": str})
fold_models = pickle.load(open(STAGE4 / "fold_models.pkl", "rb"))
p = oof[oof.model == "XGBoost"].pivot_table(index="county_fips", columns="set", values="prob", aggfunc="mean")
f = d.set_index("county_fips").join(p)
f["p"], f["p_geo"], f["p_infra"] = f["+ AI policy"], f["+ Geography"], f["Infrastructure"]
f["resid"] = f[TARGET] - f.p
f["policy_shift"] = f.p - f.p_geo

# facility detail from the atlas (all three layers), for the size of what the model missed
atlas = pd.concat([gpd.read_file(ROOT / "raw data" / "im3_open_source_data_center_atlas_v2026.02.09" /
                                 "im3_open_source_data_center_atlas_v2026.02.09.gpkg", layer=lyr).drop(columns="geometry")
                   for lyr in ["building", "campus", "point"]], ignore_index=True)

atlas["county_fips"] = atlas.state_id.astype(str).str.zfill(2) + atlas.county_id.astype(str).str.zfill(3)
fac = atlas.groupby("county_fips").agg(max_sqft=("sqft", "max"), has_campus=("type", lambda t: (t == "campus").any()),
                                       operators=("operator", lambda o: ", ".join(sorted(o.dropna().unique())[:4])))
f = f.join(fac)

# OOF SHAP: average of the three seeds' fold models (each county is in exactly one test fold per seed)
shap_rows, base_vals = [], {}
for (seed, k, set_name), model in fold_models.items():
    if set_name != "+ AI policy":
        continue
    te = [s for s in SPLITS if s[0] == seed and s[1] == k][0][3]
    Xt = pd.DataFrame(model.named_steps["prep"].transform(d[FULL].iloc[te]), columns=FULL)
    ex = shap.TreeExplainer(model.named_steps["m"])
    shap_rows.append(pd.DataFrame(ex.shap_values(Xt), columns=FULL)
                     .assign(county_fips=d.county_fips.values[te], base_value=float(np.ravel(ex.expected_value)[0])))
oof_shap = pd.concat(shap_rows).groupby("county_fips")[FULL + ["base_value"]].mean()
oof_shap.to_csv(STAGE4 / "oof_shap.csv")

### Testing the 6 failure hypotheses

In [ ]:
f["pop_quintile"] = pd.qcut(f.population_2024, 5, labels=["Q1 (smallest)", "Q2", "Q3", "Q4", "Q5 (largest)"])
FLAG = 0.20                                            # a screening threshold, see Table A5
f["flagged"] = f.p >= FLAG

by_pop = f.groupby("pop_quintile", observed=True).apply(lambda g: pd.Series({
    "counties": len(g), "with_dc": int(g[TARGET].sum()), "observed_rate": g[TARGET].mean(), "mean_predicted": g.p.mean(),
    "recall_at_0.20": (g.flagged & (g[TARGET] == 1)).sum() / max(g[TARGET].sum(), 1),
    "precision_at_0.20": (g.flagged & (g[TARGET] == 1)).sum() / max(g.flagged.sum(), 1)}), include_groups=False)
by_pop.to_csv(STAGE4 / "residuals_by_population.csv")

rho_pop, p_pop = spearmanr(f.resid, np.log10(f.population_2024))
between = f.groupby(GROUP).resid.transform("mean").var() / f.resid.var()
same_sign = f.groupby(GROUP).policy_shift.apply(lambda s: max((s > 0).mean(), (s < 0).mean()))
artifact = f.commercial_electricity_rates_dollar_per_kwh_mean.gt(1) | f.industrial_electricity_rates_dollar_per_kwh_mean.gt(1)
tp, fn = f[(f[TARGET] == 1) & f.flagged], f[(f[TARGET] == 1) & ~f.flagged]
top_fp = f[f[TARGET] == 0].sort_values("p", ascending=False).head(20)
big = f[f.max_sqft >= 1_000_000]

hyp = pd.DataFrame([
    ("Label noise", f"The 20 highest-scoring counties without an atlas facility have median population "
     f"{top_fp.population_2024.median():,.0f} (all counties {f.population_2024.median():,.0f}); "
     f"{(top_fp.pop_quintile == 'Q5 (largest)').sum()} of 20 are in the largest population quintile"),
    
    ("Policy is constant within a state", f"{between:.1%} of residual variance lies between states; in the median state "
     f"{same_sign.median():.0%} of counties move the same direction when policy is added"),
    
    ("Missing demand features", f"Largest population quintile: observed rate {by_pop.observed_rate.iloc[-1]:.1%} vs predicted "
     f"{by_pop.mean_predicted.iloc[-1]:.1%}; smallest three quintiles: {f[f.pop_quintile.isin(by_pop.index[:3])][TARGET].mean():.1%} "
     f"vs {f[f.pop_quintile.isin(by_pop.index[:3])].p.mean():.1%}. Adding log population and density lifts XGBoost PR-AUC from "
     f"{sensitivity.policy_set_mean.iloc[0]:.3f} to {sensitivity.policy_set_mean.iloc[4]:.3f}"),
    
    ("Rural hyperscale outliers", f"At p >= {FLAG}, {len(fn)} of {int(f[TARGET].sum())} positive counties are missed; median "
     f"largest facility {fn.max_sqft.median():,.0f} sq ft vs {tp.max_sqft.median():,.0f} for caught counties. Of the "
     f"{len(big)} counties whose largest facility is 1 million sq ft or more, {(~big.flagged).sum()} are missed; "
     f"{(~big.flagged & (big.population_2024 < 100_000)).sum()} of those are rural (under 100,000 people)"),
    
    ("Temporal mismatch", "Not testable: the atlas has no build dates"),
    
    ("Raster artifacts", f"{artifact.sum()} counties with impossible electricity rates: {int(f.loc[artifact, TARGET].sum())} "
     f"positive ({', '.join(f[artifact & (f[TARGET] == 1)].county_name.str.title() + ' ' + f[artifact & (f[TARGET] == 1)][GROUP])}, "
     f"p = {', '.join(f[artifact & (f[TARGET] == 1)].p.round(2).astype(str))}); mean |residual| "
     f"{f.loc[artifact, 'resid'].abs().mean():.3f} vs {f.loc[~artifact, 'resid'].abs().mean():.3f} elsewhere; electricity alone "
     f"scores PR-AUC {ablation.alone_mean.iloc[1]:.3f} (no skill {y.mean():.3f})"),
], columns=["hypothesis", "finding"])

hyp.to_csv(STAGE4 / "failure_hypotheses.csv", index=False)
for h, t in hyp.values:
    print(f"- {h}: {t}")

by_pop.round(3)

- Label noise: The 20 highest-scoring counties without an atlas facility have median population 485,604 (all counties 26,172); 17 of 20 are in the largest population quintile
- Policy is constant within a state: 4.5% of residual variance lies between states; in the median state 78% of counties move the same direction when policy is added
- Missing demand features: Largest population quintile: observed rate 30.5% vs predicted 20.8%; smallest three quintiles: 1.0% vs 4.1%. Adding log population and density lifts XGBoost PR-AUC from 0.378 to 0.519
- Rural hyperscale outliers: At p >= 0.2, 134 of 245 positive counties are missed; median largest facility 119,102 sq ft vs 143,011 for caught counties. Of the 52 counties whose largest facility is 1 million sq ft or more, 36 are missed; 18 of those are rural (under 100,000 people)
- Temporal mismatch: Not testable: the atlas has no build dates
- Raster artifacts: 15 counties with impossible electricity rates: 2 positive (Aiken SC, Berkeley SC, 

,counties,with_dc,observed_rate,mean_predicted,recall_at_0.20,precision_at_0.20
pop_quintile,,,,,,
Q1 (smallest),620.0,4.0,0.006,0.029,0.000,0.000
Q2,620.0,7.0,0.011,0.041,0.000,0.000
Q3,619.0,8.0,0.013,0.053,0.000,0.000
Q4,620.0,37.0,0.060,0.073,0.135,0.128
Q5 (largest),620.0,189.0,0.305,0.208,0.561,0.422


>The model systematically under predicts for small counties and under predicts even the largest ones relative to how often they actually get sited (predicted 20.8% vs. observed 30.5% in the top population quintile), and it catches essentially none of the real sites in the three smallest quintiles. The specific failure modes: some “false positives” (like Delaware, PA) look like label noise but are real sites just missing from the dataset. “**Missed**” sites are often rural hyperscale facilities (like Crook, OR an Apple/Meta campus in a county of 27,000 people) that break the model’s population driven pattern. For at least one real site (Fulton, GA), adding policy features actively pulled a correct prediction but in the wrong direction.

### Three failure cases (Fig. A3)

One per category, chosen by rule:
1. **Confident false positive**: the highest probability county with no atlas facility.
2. **Missed campus**: the lowest probability county that holds an atlas campus.
3. **Policy moved it the wrong way**: the largest shift toward the wrong answer when policy is added.

In [ ]:
show = ["county_name", GROUP, TARGET, "atlas_record_count", "max_sqft", "operators", "p_infra", "p_geo", "p",
        "policy_shift", "population_2024"]

neg, pos = f[f[TARGET] == 0], f[f[TARGET] == 1]
case1 = neg.p.idxmax()
case2 = pos[pos.has_campus == True].p.idxmin()
wrong = f[((f[TARGET] == 1) & (f.policy_shift < 0)) | ((f[TARGET] == 0) & (f.policy_shift > 0))]
case3 = wrong.drop(index=[case1, case2], errors="ignore").policy_shift.abs().idxmax()
cases = pd.DataFrame({"case": ["Confident false positive", "Missed campus", "Policy moved it the wrong way"],
                      "county_fips": [case1, case2, case3]}).set_index("county_fips").join(f[show])

contrib = (oof_shap.loc[cases.index, FULL].stack().rename("shap").reset_index()
           .rename(columns={"level_1": "feature"}))
raw = d.set_index("county_fips").loc[cases.index, FULL].stack().rename("value").reset_index().rename(columns={"level_1": "feature"})
contrib = contrib.merge(raw, on=["county_fips", "feature"]).merge(oof_shap[["base_value"]], left_on="county_fips", right_index=True)
cases.to_csv(STAGE4 / "failure_cases.csv")
contrib.to_csv(STAGE4 / "failure_shap.csv", index=False)

for fips, r in cases.iterrows():
    top = contrib[contrib.county_fips == fips].assign(a=lambda t: t.shap.abs()).nlargest(4, "a")
    print(f"{r.case}: {r.county_name}, {r[GROUP]} ({fips}) label {r[TARGET]}, p {r.p:.3f} (infra {r.p_infra:.3f}, "
          f"geo {r.p_geo:.3f}); top SHAP {dict(zip(top.feature, top.shap.round(2)))}")

cases.round(3)

Confident false positive: DELAWARE, PA (42045) label 0, p 0.926 (infra 0.786, geo 0.733); top SHAP {'land_value_dollar_per_sqft_mean': 1.9500000476837158, 'distance_to_substation_mean': 0.6800000071525574, 'real_property_tax_rate_mean': 0.23999999463558197, 'commercial_electricity_rates_dollar_per_kwh_mean': -0.1599999964237213}
Missed campus: CROOK, OR (41013) label 1, p 0.009 (infra 0.019, geo 0.009); top SHAP {'land_value_dollar_per_sqft_mean': -1.309999942779541, 'distance_to_substation_mean': -0.5400000214576721, 'commercial_electricity_rates_dollar_per_kwh_mean': -0.3700000047683716, 'composite_siting_suitability_mean': -0.23000000417232513}
Policy moved it the wrong way: FULTON, GA (13121) label 1, p 0.540 (infra 0.788, geo 0.803); top SHAP {'land_value_dollar_per_sqft_mean': 1.559999942779541, 'composite_siting_suitability_mean': 0.44999998807907104, 'distance_to_substation_mean': 0.4099999964237213, 'pol_pass_rate': -0.2199999988079071}


,case,county_name,state_abbr,has_data_center,atlas_record_count,max_sqft,operators,p_infra,p_geo,p,policy_shift,population_2024
county_fips,,,,,,,,,,,,
42045,Confident false positive,DELAWARE,PA,0,0,NaN,NaN,0.786,0.733,0.926,0.193,584882
41013,Missed campus,CROOK,OR,1,13,17379079.0,"Apple, Meta",0.019,0.009,0.009,-0.000,27336
13121,Policy moved it the wrong way,FULTON,GA,1,16,1294301.0,"Ascent, CoreSite, Digital Realty, Flexential",0.788,0.803,0.540,-0.263,1090354


### Top confident false positives

In [11]:
cols_show = ["county_name", GROUP, TARGET, "atlas_record_count", "p_infra", "p_geo", "p", "population_2024", "max_sqft"]
f.sort_values("p", ascending=False).loc[lambda t: t[TARGET] == 0, cols_show].head(12).round(3)

,county_name,state_abbr,has_data_center,atlas_record_count,p_infra,p_geo,p,population_2024,max_sqft
county_fips,,,,,,,,,
42045,DELAWARE,PA,0,0,0.786,0.733,0.926,584882,NaN
42071,LANCASTER,PA,0,0,0.599,0.658,0.833,563293,NaN
47093,KNOX,TN,0,0,0.589,0.538,0.710,506748,NaN
42029,CHESTER,PA,0,0,0.672,0.733,0.710,560745,NaN
20209,WYANDOTTE,KS,0,0,0.484,0.633,0.692,169418,NaN
37129,NEW HANOVER,NC,0,0,0.451,0.500,0.619,243333,NaN
36059,NASSAU,NY,0,0,0.601,0.581,0.583,1392438,NaN
53029,ISLAND,WA,0,0,0.607,0.650,0.578,86478,NaN
21067,FAYETTE,KY,0,0,0.555,0.627,0.576,329437,NaN


>12 counties the model was most sure about that turned out to have no recorded data center. Possibly there are actual data centers here but the data isn't up to date?

### Top missed real sites

In [12]:
f.sort_values("p").loc[lambda t: t[TARGET] == 1, cols_show + ["has_campus", "operators"]].head(12).round(3)

,county_name,state_abbr,has_data_center,atlas_record_count,p_infra,p_geo,p,population_2024,max_sqft,has_campus,operators
county_fips,,,,,,,,,,,
41013,CROOK,OR,1,13,0.019,0.009,0.009,27336,17379079.0,True,"Apple, Meta"
38021,DICKEY,ND,1,1,0.017,0.013,0.013,4930,2706037.0,True,Applied Digital
41065,WASCO,OR,1,2,0.018,0.014,0.017,26507,1487396.0,True,Google
54065,MORGAN,WV,1,1,0.005,0.010,0.017,17780,NaN,False,
04019,PIMA,AZ,1,1,0.027,0.050,0.018,1080149,14373.0,False,DirecTV
49047,UINTAH,UT,1,1,0.028,0.019,0.018,38307,12971.0,False,
47079,HENRY,TN,1,1,0.018,0.015,0.021,32883,85491.0,True,
41049,MORROW,OR,1,28,0.029,0.015,0.021,12360,284033.0,False,Amazon Web Services
38017,CASS,ND,1,1,0.028,0.026,0.022,200945,25931.0,False,


>The 12 real data center counties the model was least confident about. Mostly small, rural counties with one giant facility (Apple, Meta, Google, AWS). There seems to be a specific model blindspot here.

### Top policy shifted counties

In [13]:
wrong.reindex(wrong.policy_shift.abs().sort_values(ascending=False).index)[cols_show + ["policy_shift"]].head(12).round(3)

,county_name,state_abbr,has_data_center,atlas_record_count,p_infra,p_geo,p,population_2024,max_sqft,policy_shift
county_fips,,,,,,,,,,
13121,FULTON,GA,1,16,0.788,0.803,0.540,1090354,1294301.0,-0.263
34023,MIDDLESEX,NJ,1,13,0.917,0.969,0.722,890119,829908.0,-0.247
22071,ORLEANS,LA,0,0,0.430,0.287,0.522,362701,NaN,0.235
39057,GREENE,OH,0,0,0.279,0.227,0.436,172347,NaN,0.209
13067,COBB,GA,1,3,0.707,0.659,0.459,787538,313479.0,-0.200
42045,DELAWARE,PA,0,0,0.786,0.733,0.926,584882,NaN,0.193
36087,ROCKLAND,NY,1,4,0.875,0.761,0.583,348144,262772.0,-0.178
42071,LANCASTER,PA,0,0,0.599,0.658,0.833,563293,NaN,0.176
51650,HAMPTON,VA,0,0,0.477,0.395,0.570,137596,NaN,0.175


>12 counties where adding AI policy moved the prediction the most (in both directions). *Policy can swing individual counties a lot, even though its average effect across all counties is small.*